# Computer Vision Track · CV Masterclass 05: Segmentation & Localization

Welcome to Notebook 05. Object detection draws a box. Segmentation is pixel-perfect intelligence. It classifies every single pixel in an image into a distinct semantic class.

We will explore the legendary U-Net, the mechanics of Decoder upsampling, Atrous Pyramids (DeepLab), and how foundation models like **SAM** redefined the field with zero-shot prompting.

---

## 🎓 Socratic Deep Check
Ponder this question before we begin:

> *"In a U-Net, the Encoder compresses the image into a tiny bottleneck. The Decoder expands it back. What exact mathematical spatial information is permanently destroyed during compression that forces the architecture to use 'Skip Connections' to restore sharp boundaries?"*

---

1. **Semantic vs Instance Segmentation:** Blobs vs Entities.
2. **U-Net & Skip Connections:** Restoring lost spatial coordinates.
3. **Upsampling Mechanics:** Transposed Conv vs Interpolation.
4. **DeepLab & ASPP:** Atrous (Dilated) pyramids for multi-scale context.
5. **Loss Functions:** Dice, Tversky, and Boundary-Aware loss.
6. **Mask R-CNN:** Instance segmentation via RoI Align.
7. **Panoptic Segmentation:** Dealing with 'Things' and 'Stuff'.
8. **Segment Anything Model (SAM):** Promptable zero-shot segmentation.
9. **Mask2Former:** Universal mask classification.
10. **Deep Socratic Synthesis.**


## 📑 Table of Contents

* [🎓 Socratic Deep Check](#socratic-deep-check)
* [1. Semantic vs Instance Segmentation](#1-semantic-vs-instance-segmentation)
* [2. U-Net: The Medical Gold Standard](#2-u-net-the-medical-gold-standard)
* [3. Upsampling: Transposed Conv vs Interpolation](#3-upsampling-transposed-conv-vs-interpolation)
* [4. DeepLab & ASPP (Atrous Pyramid)](#4-deeplab-aspp-atrous-pyramid)
* [5. Evaluation & Losses (mIoU, Dice, Tversky)](#5-evaluation-losses-miou-dice-tversky)
* [6. Mask R-CNN: Boxes to Masks](#6-mask-r-cnn-boxes-to-masks)
* [7. Panoptic Segmentation](#7-panoptic-segmentation)
* [8. SAM: Segment Anything (Foundation Vision)](#8-sam-segment-anything-foundation-vision)
* [9. Mask2Former: One Architecture to Rule Them All](#9-mask2former-one-architecture-to-rule-them-all)
* [10. 🎓 Deep Socratic Synthesis](#10-deep-socratic-synthesis)


### 🎓 Junior to Senior: Concept Bridge

**Junior:** Treats Semantic Segmentation as an entirely different problem space requiring entirely specific models.

**Senior:** Views segmentation merely as pixel-wise dense classification. Uses U-Net style Encoder-Decoder architectures to recapture precise spatial dimensions that were lost during standard feature poolings.

---


### 🎓 Junior to Senior: Concept Bridge

**Junior:** Treats Semantic Segmentation as an entirely different problem space requiring entirely specific models.

**Senior:** Views segmentation merely as pixel-wise dense classification. Uses U-Net style Encoder-Decoder architectures to recapture precise spatial dimensions that were lost during standard feature poolings.

---


### 🎓 Junior to Senior: Concept Bridge

**Junior:** Treats Semantic Segmentation as an entirely different problem space requiring entirely specific models.

**Senior:** Views segmentation merely as pixel-wise dense classification. Uses U-Net style Encoder-Decoder architectures to recapture precise spatial dimensions that were lost during standard feature poolings.

---


## 1. Semantic vs Instance Segmentation

- **Semantic**: Classify every pixel category (e.g., all sheep are 'sheep' color). No entity separation.
- **Instance**: Distinguish individual objects (e.g., Sheep #1 is red, Sheep #2 is blue). Usually Object Detection + Masking.
- **Panoptic**: The union. Semantic for background ('Stuff': grass, sky) and Instance for foreground ('Things': cars, people).

## 2. U-Net: The Medical Gold Standard

U-Net uses **Skip Connections** to concatenate high-resolution features from the Encoder directly into the Decoder. 
**Why?** Max-pooling destroys the exact $(x, y)$ coordinates of edges. The deep bottleneck knows *what* it's looking at, but not *where* the boundaries are. Skip connections provide the "spatial map" for the decoder to redraw the mask at pixel precision.

In [ ]:
import torch
import torch.nn as nn

class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class SimpleUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = UNetBlock(3, 64)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = UNetBlock(64, 128)
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec1 = UNetBlock(128 + 64, 64) # 128 (up) + 64 (skip)
        self.head = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        s1 = self.enc1(x)
        b = self.bottleneck(self.pool(s1))
        d = self.up(b)
        # Concatenate Skip Connection
        out = self.dec1(torch.cat([d, s1], dim=1))
        return self.head(out)

dummy = torch.randn(1, 3, 224, 224)
model = SimpleUNet()
print(f"U-Net Output: {model(dummy).shape}")


## 3. Upsampling: Transposed Conv vs Interpolation

- **Transposed Conv**: Learns to splat pixels back up. Can cause "Checkerboard Artifacts" due to uneven overlapping.
- **Interpolation + Conv**: Resize with bilinear math, then refine with a standard conv. This is the modern, more stable standard for clean masks.

## 4. DeepLab & ASPP (Atrous Pyramid)

**ASPP (Atrous Spatial Pyramid Pooling)** uses parallel **Dilated Convolutions** with different rates (6, 12, 18). This allows the model to see tiny microscopic textures and massive macroscopic context (like a whole building) in a single pass without increasing parameters.

## 5. Evaluation & Losses (mIoU, Dice, Tversky)

- **mIoU (Mean Intersection over Union)**: Standard metric. Accuracy is a lie in segmentation (99% of pixels can be background).
- **Dice Loss**: $L = 1 - \frac{2|X \cap Y|}{|X| + |Y|}$. Robust to small objects in large backgrounds.
- **Tversky Loss**: Allows weighting False Negatives (missing a tumor) more heavily than False Positives.

In [ ]:
def dice_loss(pred, target, smooth=1e-6):
    intersection = (pred * target).sum()
    return 1 - (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)


## 6. Mask R-CNN: Boxes to Masks

Mask R-CNN adds a mask head to Faster R-CNN. It uses **RoI Align** (Bilinear sub-pixel extraction) instead of RoI Pool to prevent the rounding/quantization errors that destroy mask boundary precision.

## 7. Panoptic Segmentation

Evaluated using **PQ (Panoptic Quality)**. It explicitly penalizes both identification errors (wrong segments) and geometric errors (poor mask overlap).

## 8. SAM: Segment Anything (Foundation Vision)

Meta's **Segment Anything Model (SAM)** redefined the field:
- **Promptable**: Take a point, box, or text and get a mask instantly.
- **Zero-shot**: Works on medical, satellite, and underwater data without any extra training.
- **Architecture**: Massive ViT Image Encoder + tiny prompt encoder + lightweight transformer mask decoder.

## 9. Mask2Former: One Architecture to Rule Them All

Instead of separate models for Semantic, Instance, and Panoptic, **Mask2Former** treats all three as a **Universal Mask Classification** problem. It uses learned queries (like DETR) to attend to regions and output masks directly.

---
## ✅ Knowledge Check

### Q1: What is the difference between Semantic and Instance Segmentation?
<details><summary>👉 Answer</summary>

Semantic segmentation classifies every pixel into a class (e.g. 'car', 'road'). Instance segmentation identifies individual occurrences of classes (e.g. 'car 1', 'car 2').
</details>

### Q2: Why are Skip Connections crucial in the U-Net architecture for segmentation?
<details><summary>👉 Answer</summary>

The encoder drastically shrinks the spatial dimensions to learn 'what' is in the image. Skip connections pass the high-resolution, early layers directly over to the decoder to tell it 'where' the edges precisely are.
</details>

---
## 🔨 Practical Practice

### Exercise 1
Dice Loss Setup: Implement soft Dice Loss in PyTorch to combat extreme class imbalance in medical image segmentation.

### Exercise 2
Transposed Convs: Utilize `nn.ConvTranspose2d` to upsample a feature map back to its original image resolution.


## 10. 🎓 Deep Socratic Synthesis

**Q:** *Why is SAM considered a 'Foundation Model' while U-Net is just a 'Medical Network'?*

**Ponder deeply:** U-Net learns the specific bias of the dataset it's trained on (e.g., 'cell shapes'). It cannot segment a car if trained on cells. SAM, trained on **1.1 Billion masks**, learns the universal concept of 'what is a bounded object'. It doesn't care about the class; it cares about the grouping. This decoupling of 'finding an object' and 'classifying an object' is the paradigm shift to generic vision intelligence.